In [ ]:
#Creating Folium Model for Visualization
#Importing libraries for mapping
import folium 
from folium.plugins import HeatMap, TimestampedGeoJson
from IPython.display import display, HTML



#Mapping incidents for dataset
map_data = folium.Map(location=[full_df['Latitude'].mean(), full_df['Longitude'].mean()], zoom_start=3,tiles='cartodbpositron')

#Preparing date data 
full_df['Day of Year']= pd.to_datetime('2023-01-01')+pd.to_timedelta(full_df['Day of Year'])
full_df ['month start'] = full_df['Day of Year'].dt.to_period('M').dt.to_timestamp()


#Adding markers 
for _,row in full_df.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    species =row['Species Code']
    attack =row['Attack']

    if attack ==1: 
        if species == 0:
            marker_color = 'black'
            icon_color = 'white'
            name = 'Black Bear'
        elif species == 1:
            marker_color = 'darkred'
            icon_color = 'white'
            name = 'Grizzly Bear'

        folium.Marker(
            location= [lat, lon], 
            popup=f"{name}", 
            icon = folium.Icon(color=marker_color, icon_color= icon_color, icon='bear')
        ).add_to(map_data)

features=[]
for _,row in results_df.iterrows():
    features.append({
        'type':'Feature',
        'geometry': {'type':'Point', 'coordinates':[row['Longitude'], row['Latitude']]},
        'properties':{
            'time':row['month start'].strftime('%Y-%m-%d'),
            'popup':f"{'Black Bear' if row['Species Code']==0 else 'Grizzly Bear'}, Probability: {row['Probabilities']:.2f}",
            'icon': 'circle',
            'iconstyle':{
              'fillColor': 'gray' if row['Species Code']==0 else 'darkred',
              'fillOpacity': 0.8, 
              'stroke': 'true',
              'radius':5
            }

        }
    })

geojson = {'type':'FeatureCollection', 'features': features}
TimestampedGeoJson(
    geojson,
    period='P1M',
    add_last_point= True,
    auto_play= False,
    loop= False,
    max_speed =1,
    loop_button=True, 
    date_options='YYYY-MM',
    time_slider_drag_update= True

).add_to(map_data)
HTML(map_data._repr_html_())

In [ ]:
#Mapping Model Predictions Using Folium
map_pred = folium.Map(location=[results_df['Latitude'].mean(), results_df['Longitude'].mean()], zoom_start=3,tiles='cartodbpositron')

#Creating Feature Groups 
fg_both =folium.FeatureGroup (name='All Bears', show=True)
fg_black=folium.FeatureGroup (name = 'Black Bears', show= True)
fg_grizzly = folium.FeatureGroup (name = 'Grizzly Bears', show= True)


#Collect heatmap points 
heat_bears=[]
heat_black=[]
heat_grizzly=[]

#Preparing date data 
results_df['Day of Year']= pd.to_datetime('2023-01-01')+pd.to_timedelta(results_df['Day of Year'])
results_df ['month start'] = results_df['Day of Year'].dt.to_period('M').dt.to_timestamp()

#Adding markers by iterating through dataframe
for _,row in results_df.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    species =row['Species Code']
    prediction =row['Predictions']
    probability = row['Probabilities']
    date = row['month start']
    
    #Only plotting attacks
    if prediction ==1: 
        #Species specific colouring and name
        if species == 0: #black bears
            marker_color = 'black'
            icon_color = 'white'
            name = 'Black Bear'
            heat_black.append([lat, lon, probability])

        else: #grizzly bears
            marker_color = 'darkred'
            icon_color = 'white'
            name = 'Grizzly Bear'
            heat_grizzly.append([lat, lon, probability])
        heat_bears.append([lat, lon, probability])

        #Creating markers for each feature group
        if species == 0: 
            fg_black.add_child(
            folium.Marker(
                location= [lat, lon], 
                popup=f"{name}", 
                icon = folium.Icon(color=marker_color, icon_color= icon_color, icon='info-sign')
            )
        )
        else:
            fg_grizzly.add_child(
            folium.Marker(
                location= [lat, lon], 
                popup=f"{name}", 
                icon = folium.Icon(color=marker_color, icon_color= icon_color, icon='info-sign')
            )
        )
        fg_both.add_child(
            folium.Marker(
                location= [lat, lon], 
                popup=f"{name}", 
                icon = folium.Icon(color=marker_color, icon_color= icon_color, icon='info-sign')
            )
        )
        
        
#Creating heatmap for each feature group
fg_black.add_child(
    HeatMap(heat_black, min_opacity=0.2, radius=8, blur=15)
)    

fg_grizzly.add_child(
    HeatMap(heat_grizzly, min_opacity=0.2, radius=8, blur=15)
)

fg_both.add_child(
    HeatMap(heat_bears, min_opacity=0.2, radius=8, blur=15)
)

#Adding each feature group to map
fg_black.add_to(map_pred)
fg_grizzly.add_to(map_pred)
fg_both.add_to(map_pred)

#Adding layer control
folium.LayerControl(collapsed=False).add_to(map_pred)

#Making time slider (monthly)
features=[]
for _,row in results_df.iterrows():
    features.append({
        'type':'Feature',
        'geometry': {'type':'Point', 'coordinates':[row['Longitude'], row['Latitude']]},
        'properties':{
            'time':row['month start'].strftime('%Y-%m-%d'),
            'popup':f"{'Black Bear' if row['Species Code']==0 else 'Grizzly Bear'}, Probability: {row['Probabilities']:.2f}",
            'icon': 'circle',
            'iconstyle':{
              'fillColor': 'gray' if row['Species Code']==0 else 'darkred',
              'fillOpacity': 0.8, 
              'stroke': 'true',
              'radius':5
            }

        }
    })

geojson = {'type':'FeatureCollection', 'features': features}
TimestampedGeoJson(
    geojson,
    period='P1M',
    add_last_point= True,
    auto_play= False,
    loop= False,
    max_speed =1,
    loop_button=True, 
    date_options='YYYY-MM',
    time_slider_drag_update= True

).add_to(map_pred)



#Display map for predicitons
map_pred